# Fit stochastic volatility models

- Generate a lot of time series from stochastic volatility models, with known parameters
- Compute a few statistics ("generalized moments) of those time series (first moments and autocorrelations of $x$, $|x|$, $x^2$, $\log|x|$, and their first autocorrelations)
- Train a machine learning model to recover the model parameters from those statistics
- Save the model for later use

This may not be very precise, but I just want to know what the "typical values" of those parameters are.

In [ ]:
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from statsmodels.tsa.stattools import acf
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor

from functions import stochastic_volatility_returns, sv_features

for message in [ 
    'No further splits with positive gain, best gain: -inf',
    'No further splits with positive gain',
]:
    warnings.filterwarnings( 'ignore', message=message )

In [ ]:
ESTIMATORS   = 500  # 100 may be enough
SAMPLES      = 10_000
REPLICATIONS = 100 

In [ ]:
def sv_features(r, K=5):
    r = r - np.nanmean(r)
    r2 = r**2
    abs_r = np.abs(r)
    log_r2 = np.log(r2 + 1e-8)

    return np.r_[
        np.log(np.var(r) + 1e-12),
        np.mean(log_r2),
        np.var(log_r2),
        np.mean(r2**2) / (np.mean(r2)**2 + 1e-12),
        acf(r2, nlags=K, fft=True)[1:],
        acf(abs_r, nlags=K, fft=True)[1:],
        acf(log_r2, nlags=K, fft=True)[1:]
    ]

In [ ]:
import ray

ray.init(ignore_reinit_error=True, include_dashboard=False)

@ray.remote
def simulate_one(mu, phi, sigma, replications, n=1000, burnin=1000, df=5):
    records = []
    for _ in range(replications):
        innovations = np.random.standard_t(df=df, size=n+burnin)
        y = stochastic_volatility_returns(n, mu, phi, sigma, innovations)[0]
        input_features = sv_features(y)
        input_dict = {f'feature_{i}': input_features[i] for i in range(len(input_features))}
        output = {'mu': mu, 'phi': phi, 'sigma': sigma}
        records.append(input_dict | output)
    return records

d = []
for _ in tqdm(range(SAMPLES)):
    mu = np.random.uniform(-5, 1)
    phi = np.random.uniform(-1, 1)
    sigma = np.exp(np.random.uniform(-2, 2))
    d.append( simulate_one.remote(mu, phi, sigma, REPLICATIONS) )

d = [ ray.get(u) for u in tqdm(d) ]
d = [item for sublist in d for item in sublist]
d = pd.DataFrame(d)
d = d.groupby(['mu', 'phi', 'sigma']).mean().reset_index()
d.tail()

In [ ]:
if False:
        
    d = []
    for _ in tqdm( range( SAMPLES ) ):
            
        mu = np.random.uniform( -5, 1 )
        phi = np.random.uniform( -1, 1 )
        sigma = np.exp( np.random.uniform( -2, 2 ) )
        output = { 'mu': mu, 'phi': phi, 'sigma': sigma }

        for _ in range( REPLICATIONS ):
            n = 1000
            burnin = 1000
            innovations = np.random.standard_t(df = 5, size = n + burnin)
            y = stochastic_volatility_returns( n, mu, phi, sigma, innovations )[0]
            input = sv_features( y )
            input = { f'feature_{i}': input[i] for i in range(len(input)) }
            d.append( input | output )
    d = pd.DataFrame( d )
    d = d.groupby( ['mu', 'phi', 'sigma'] ).mean().reset_index()


In [ ]:
if False: 
    
    X = d.drop( columns = ['mu', 'phi', 'sigma'] )
    y = d[ ['mu', 'phi', 'sigma'] ]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    models = {}
    for target in ['mu', 'phi', 'sigma']:
        print( f'Training {target} model...' )
        model = LGBMRegressor(
            n_estimators  = 10,
            learning_rate = 0.1,
            max_depth     = 6,
            random_state  = 42,
            verbosity     = -1,
        )
        model.fit(X_train, y_train[target])
        models[target] = model

    fig, axs = plt.subplots( 1, 3, figsize = (9,3), layout = 'constrained', dpi = 300 )
    for ax, target in zip( axs, ['mu', 'phi', 'sigma'] ):
        ax.scatter(
            models[target].predict( X_test ),
            y_test[target],
            alpha = 0.1,
        )
        ax.set_title( f'{target}' )
        ax.set_xlabel( 'Predicted' )
        ax.set_ylabel( 'Actual' )
    plt.show()

In [ ]:
X = d.drop( columns = ['mu', 'phi', 'sigma'] )
y = d[ ['mu', 'phi', 'sigma'] ]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LGBMRegressor(
    n_estimators  = ESTIMATORS,
    learning_rate = 0.1,
    max_depth     = 6,
    random_state  = 42,
    verbosity     = -1,
)
model = MultiOutputRegressor( model )
model.fit( X_train, y_train )

y_hat = model.predict( X_test )
y_hat = pd.DataFrame( y_hat, columns = y_test.columns )

In [ ]:
filename = f'../cache/stochastic_volatility_features_to_parameters_{ESTIMATORS}_{SAMPLES}_{REPLICATIONS}.pkl'
with open(filename, 'wb') as f:
    pickle.dump(model, f)

with open(filename, 'rb') as f:
    model = pickle.load(f)

In [ ]:
fig, axs = plt.subplots( 1, 3, figsize = (9,3), layout = 'constrained', dpi = 300 )
for ax, target in zip( axs, ['mu', 'phi', 'sigma'] ):
    ax.scatter(
        y_hat[target],
        y_test[target],
        alpha = 0.2,
        s = 5,
    )
    ax.set_title( f'{target}' )
    ax.set_xlabel( 'Predicted' )
    ax.set_ylabel( 'Actual' )
plt.show()

In [ ]:
returns = pd.read_parquet( "../cache/yfinance_returns.parquet")
returns = np.log( returns.ffill() ).diff().dropna()
returns[:] = np.where( returns == 0, np.nan, returns )
r = {}
for id in returns.columns: 
    x = returns[id].dropna()
    if len(x) < 20: 
        continue
    x = sv_features( x )
    r[id] = x
r = pd.DataFrame( r ).T
r = pd.DataFrame( model.predict(r), index = r.index, columns = ['mu', 'phi', 'sigma'] )

fig, axs = plt.subplots( 1, 3, figsize = (9,3), layout = 'constrained', dpi = 300 )
for ax, target in zip( axs, ['mu', 'phi', 'sigma'] ):
    ax.hist( r[target], bins = 10)
    ax.set_xlabel( f'{target}' )
plt.show()